<div align="center">

# LOSO Model

</div>

In [10]:
from pathlib import Path
import h5py
import numpy as np

# 1. Load Files

In [16]:
# Seizure files
feature_dir = Path("Features/Seizure")

files = sorted(feature_dir.glob("chb01_*_seizure.mat"))

print(len(files))
for f in files:
    print(f.name)

7
chb01_03_seizure.mat
chb01_04_seizure.mat
chb01_15_seizure.mat
chb01_16_seizure.mat
chb01_18_seizure.mat
chb01_21_seizure.mat
chb01_26_seizure.mat


In [15]:
# Seizure files
feature_dir = Path("Features/Non_Seizure")

files = sorted(feature_dir.glob("chb01_*_features.mat"))

print(len(files))
for f in files:
    print(f.name)

30
chb01_01_features.mat
chb01_02_features.mat
chb01_05_features.mat
chb01_06_features.mat
chb01_07_features.mat
chb01_08_features.mat
chb01_09_features.mat
chb01_10_features.mat
chb01_11_features.mat
chb01_12_features.mat
chb01_13_features.mat
chb01_14_features.mat
chb01_17_features.mat
chb01_19_features.mat
chb01_20_features.mat
chb01_22_features.mat
chb01_23_features.mat
chb01_24_features.mat
chb01_25_features.mat
chb01_27_features.mat
chb01_29_features.mat
chb01_30_features.mat
chb01_31_features.mat
chb01_32_features.mat
chb01_33_features.mat
chb01_34_features.mat
chb01_36_features.mat
chb01_37_features.mat
chb01_38_features.mat
chb01_39_features.mat


In [10]:
# Inspeact one file
with h5py.File(files[0], "r") as f:
    print(list(f.keys()))
    print(f["all_feats"].shape)
    print(f["all_labels"].shape)

['all_feats', 'all_labels', 'bands', 'feat_len', 'fres', 'morder', 'step_samps', 'win_samps']
(1437, 1530)
(1, 1437)


In [25]:
# Load records into a dictionary

records = {}

for file in files:

    with h5py.File(file, "r") as f:

        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    # If features are stored as (1530, n_windows)
    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(record_name, X.shape, y.shape)

chb01_03_seizure (1437, 1530) (1437,)
chb01_04_seizure (1437, 1530) (1437,)
chb01_15_seizure (1437, 1530) (1437,)
chb01_16_seizure (1437, 1530) (1437,)
chb01_18_seizure (1437, 1530) (1437,)
chb01_21_seizure (1437, 1530) (1437,)
chb01_26_seizure (927, 1530) (927,)


In [30]:
from dataclasses import dataclass
import numpy as np

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix
)

# =====================================================
# PARAMETERS
# =====================================================

WIN_SEC = 10.0
STEP_SEC = 2.5      # 75% overlap
K_FEATURES = 50

# =====================================================
# EVENT-LEVEL UTILITIES
# =====================================================

@dataclass
class RecordPrediction:
    record_id: str
    window_starts: np.ndarray
    window_len: float
    scores: np.ndarray
    duration_sec: float
    seizures: list


def labels_to_seizures(y, step_sec=2.5, win_sec=10.0):

    seizures = []

    ictal = (y == 1)

    i = 0
    while i < len(ictal):

        if ictal[i]:

            j = i
            while j + 1 < len(ictal) and ictal[j + 1]:
                j += 1

            onset = i * step_sec
            offset = j * step_sec + win_sec

            seizures.append((onset, offset))

            i = j + 1

        else:
            i += 1

    return seizures


def _runs_of_true(mask):

    runs = []
    i = 0

    while i < len(mask):

        if mask[i]:

            j = i

            while j + 1 < len(mask) and mask[j + 1]:
                j += 1

            runs.append((i, j))
            i = j + 1

        else:
            i += 1

    return runs


def detect_events(rec,
                  threshold=0.0,
                  k_consecutive=2,
                  refractory_sec=120):

    pred = rec.scores >= threshold

    events = []

    for s, e in _runs_of_true(pred):

        if (e - s + 1) >= k_consecutive:

            onset = rec.window_starts[s]
            offset = rec.window_starts[e] + rec.window_len

            events.append({
                "onset_sec": onset,
                "offset_sec": offset
            })

    accepted = []
    last_time = -np.inf

    for ev in events:

        if ev["onset_sec"] >= last_time + refractory_sec:

            accepted.append(ev)
            last_time = ev["onset_sec"]

    return accepted


def score_records(event_records,
                  threshold=0.0,
                  k_consecutive=2,
                  refractory_sec=120):

    total_seizures = 0
    detected_seizures = 0
    false_detections = 0
    total_hours = 0

    latencies = []

    for rec in event_records:

        total_hours += rec.duration_sec / 3600

        events = detect_events(
            rec,
            threshold,
            k_consecutive,
            refractory_sec
        )

        matched_event = [False] * len(events)

        for onset, offset in rec.seizures:

            total_seizures += 1

            seizure_detected = False

            for idx, ev in enumerate(events):

                overlap = (
                    ev["onset_sec"] <= offset and
                    ev["offset_sec"] >= onset
                )

                if overlap:

                    matched_event[idx] = True
                    seizure_detected = True

                    latencies.append(
                        ev["onset_sec"] - onset
                    )

            if seizure_detected:
                detected_seizures += 1

        false_detections += sum(
            1 for m in matched_event if not m
        )

    return {
        "sensitivity":
            detected_seizures / total_seizures
            if total_seizures else np.nan,

        "fd_per_hour":
            false_detections / total_hours
            if total_hours else np.nan,

        "median_latency_sec":
            np.median(latencies)
            if len(latencies) else np.nan,

        "n_seizures":
            total_seizures,

        "n_detected":
            detected_seizures,

        "n_false_detections":
            false_detections
    }

# =====================================================
# LORO TRAINING
# =====================================================

results = []
event_records = []

for test_record in records:

    print(f"\nTesting: {test_record}")

    # -------------------------------------------------
    # SPLIT
    # -------------------------------------------------

    X_test, y_test = records[test_record]

    X_train = np.vstack([
        records[r][0]
        for r in records
        if r != test_record
    ])

    y_train = np.concatenate([
        records[r][1]
        for r in records
        if r != test_record
    ])

    # -------------------------------------------------
    # FEATURE SELECTION
    # -------------------------------------------------

    selector = SelectKBest(
        score_func=f_classif,
        k=K_FEATURES
    )

    X_train = selector.fit_transform(
        X_train,
        y_train
    )

    X_test = selector.transform(
        X_test
    )

    # -------------------------------------------------
    # SCALING
    # -------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )

    # -------------------------------------------------
    # SVM
    # -------------------------------------------------

    model = SVC(
        kernel='rbf',
        C=0.1,
        gamma='scale',
        class_weight='balanced'
    )

    model.fit(X_train, y_train)

    # -------------------------------------------------
    # PREDICTIONS
    # -------------------------------------------------

    y_pred = model.predict(X_test)

    scores = model.decision_function(X_test)

    # -------------------------------------------------
    # SEGMENT-LEVEL METRICS
    # -------------------------------------------------

    acc = accuracy_score(y_test, y_pred)

    auc = roc_auc_score(y_test, scores)

    auprc = average_precision_score(
        y_test,
        scores
    )

    sens = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    bac = balanced_accuracy_score(
        y_test,
        y_pred
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    spec = tn / (tn + fp)

    print(f"Accuracy      : {acc:.3f}")
    print(f"ROC-AUC       : {auc:.3f}")
    print(f"AUPRC         : {auprc:.3f}")
    print(f"Sensitivity   : {sens:.3f}")
    print(f"Specificity   : {spec:.3f}")
    print(f"Balanced Acc  : {bac:.3f}")

    results.append({
        "record": test_record,
        "accuracy": acc,
        "roc_auc": auc,
        "auprc": auprc,
        "sensitivity": sens,
        "specificity": spec,
        "balanced_acc": bac
    })

    # -------------------------------------------------
    # EVENT-LEVEL OBJECT
    # -------------------------------------------------

    window_starts = (
        np.arange(len(y_test))
        * STEP_SEC
    )

    duration_sec = (
        window_starts[-1]
        + WIN_SEC
    )

    seizures = labels_to_seizures(
        y_test,
        STEP_SEC,
        WIN_SEC
    )

    event_records.append(
        RecordPrediction(
            record_id=test_record,
            window_starts=window_starts,
            window_len=WIN_SEC,
            scores=scores,
            duration_sec=duration_sec,
            seizures=seizures
        )
    )

# =====================================================
# OVERALL SEGMENT RESULTS
# =====================================================

print("\n========================")
print("SEGMENT-LEVEL RESULTS")
print("========================")

print(
    "Mean ROC-AUC:",
    np.mean([r["roc_auc"] for r in results])
)

print(
    "Mean AUPRC:",
    np.mean([r["auprc"] for r in results])
)

print(
    "Mean Sensitivity:",
    np.mean([r["sensitivity"] for r in results])
)

print(
    "Mean Specificity:",
    np.mean([r["specificity"] for r in results])
)

# =====================================================
# EVENT-LEVEL RESULTS
# =====================================================
for th in [0.0, 0.5, 1.0, 2.0]:
    for k in [2, 4, 6, 8, 10, 12]:
        out = score_records(
            event_records,
            threshold=th,
            k_consecutive=k,
            refractory_sec=120
        )
        print(
            f"th={th:3.1f}, k={k:2d} | "
            f"sens={out['sensitivity']:.3f}, "
            f"FD/h={out['fd_per_hour']:.3f}, "
            f"latency={out['median_latency_sec']:.1f}s"
        )
    print()


Testing: chb01_03_seizure
Accuracy      : 0.981
ROC-AUC       : 0.992
AUPRC         : 0.541
Sensitivity   : 0.938
Specificity   : 0.981
Balanced Acc  : 0.959

Testing: chb01_04_seizure
Accuracy      : 0.958
ROC-AUC       : 0.987
AUPRC         : 0.291
Sensitivity   : 1.000
Specificity   : 0.958
Balanced Acc  : 0.979

Testing: chb01_15_seizure
Accuracy      : 0.944
ROC-AUC       : 0.802
AUPRC         : 0.033
Sensitivity   : 0.125
Specificity   : 0.953
Balanced Acc  : 0.539

Testing: chb01_16_seizure
Accuracy      : 0.980
ROC-AUC       : 0.994
AUPRC         : 0.614
Sensitivity   : 1.000
Specificity   : 0.980
Balanced Acc  : 0.990

Testing: chb01_18_seizure
Accuracy      : 0.983
ROC-AUC       : 0.996
AUPRC         : 0.855
Sensitivity   : 1.000
Specificity   : 0.982
Balanced Acc  : 0.991

Testing: chb01_21_seizure
Accuracy      : 0.997
ROC-AUC       : 0.993
AUPRC         : 0.970
Sensitivity   : 0.947
Specificity   : 0.998
Balanced Acc  : 0.973

Testing: chb01_26_seizure
Accuracy      : 0.9